In [24]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

matches_df = pd.read_csv('data/matches.csv')
rankings_df = pd.read_csv('data/ranking.csv')


In [25]:
class PoissonModel:
    def __init__(self):
        self.avg_goals = 0
        self.team_strength = {}
        
    #average goals
    def fit(self, matches, rankings):
        
        total_goals = matches['score_1'].sum() + matches['score_2'].sum()
        self.avg_goals = total_goals / len(matches)
    
        rank_dict = dict(zip(rankings['Country_EN'], rankings['Points']))
        max_points = rankings['Points'].max()
        
       
        all_teams = list(set(matches['team_1'].unique()) | set(matches['team_2'].unique()))
        for team in all_teams:
            if team in rank_dict:
                strength = rank_dict[team] / max_points
                self.team_strength[team] = strength
            else:
                self.team_strength[team] = 0.5
    
    def predict(self, team1, team2, is_knockout=False):
        
        s1 = self.team_strength.get(team1, 0.5)
        s2 = self.team_strength.get(team2, 0.5)
        
        # expected goal
        lambda1 = self.avg_goals * (s1 / (s1 + s2)) * 2
        lambda2 = self.avg_goals * (s2 / (s1 + s2)) * 2
        
        # knockout factor (teams play more defensively)
        if is_knockout:
            lambda1 *= 0.85
            lambda2 *= 0.85
        
        
        max_goals = 8
        win1 = 0
        win2 = 0
        draw = 0
        
        for i in range(max_goals + 1):
            for j in range(max_goals + 1):
                prob = poisson.pmf(i, lambda1) * poisson.pmf(j, lambda2)
                if i > j:
                    win1 += prob
                elif i < j:
                    win2 += prob
                else:
                    draw += prob
        
        
        most_likely = (0, 0)
        max_prob = 0
        for i in range(max_goals + 1):
            for j in range(max_goals + 1):
                prob = poisson.pmf(i, lambda1) * poisson.pmf(j, lambda2)
                if prob > max_prob:
                    max_prob = prob
                    most_likely = (i, j)
        
        #  handle draws with ET and penalties
        if is_knockout and draw > 0:
            # extra time: 30% chance of a goal in ET for each team
            et_win1 = draw * 0.15  
            et_win2 = draw * 0.15  
            remaining_draw = draw - et_win1 - et_win2
            
            
            pen_win1 = remaining_draw * 0.5
            pen_win2 = remaining_draw * 0.5
            
          
            final_win1 = win1 + et_win1 + pen_win1
            final_win2 = win2 + et_win2 + pen_win2
            
            return {
                'team1': team1,
                'team2': team2,
                'xG': (lambda1, lambda2),  
                'win1_90': win1,
                'draw_90': draw,
                'win2_90': win2,
                'win1_final': final_win1,
                'win2_final': final_win2,
                'most_likely': f"{most_likely[0]}-{most_likely[1]}",
                'winner': team1 if final_win1 > final_win2 else team2
            }
        else:
            return {
                'team1': team1,
                'team2': team2,
                'xG': (lambda1, lambda2), 
                'win1': win1,
                'draw': draw,
                'win2': win2,
                'most_likely': f"{most_likely[0]}-{most_likely[1]}",
                'winner': team1 if win1 > win2 and win1 > draw else 
                          team2 if win2 > win1 and win2 > draw else 
                          'Draw'
            }


In [26]:
model = PoissonModel()
model.fit(matches_df, rankings_df)

In [31]:
model = PoissonModel()
model.fit(matches_df, rankings_df)

print("2026 WORLD CUP PREDICTIONS")


# Semi-Finals
print("\n🏆 SEMI-FINAL 1: Spain vs France")
sf1 = model.predict('Spain', 'France', is_knockout=True)
print(f"  xG: Spain {sf1['xG'][0]:.2f} - {sf1['xG'][1]:.2f} France")
print(f"  Most Likely (90 mins): {sf1['most_likely']}")
print(f"  After 90 mins: Spain {sf1['win1_90']:.1%} | Draw {sf1['draw_90']:.1%} | France {sf1['win2_90']:.1%}")
print(f"  After ET+Penalties: Spain {sf1['win1_final']:.1%} | France {sf1['win2_final']:.1%}")
print(f"  👉 Winner: {sf1['winner']}")
print('-'*60)
print("\n🏆 SEMI-FINAL 2: Argentina vs England")
sf2 = model.predict('Argentina', 'England', is_knockout=True)
print(f"  xG: Argentina {sf2['xG'][0]:.2f} - {sf2['xG'][1]:.2f} England")
print(f"  Most Likely (90 mins): {sf2['most_likely']}")
print(f"  After 90 mins: Argentina {sf2['win1_90']:.1%} | Draw {sf2['draw_90']:.1%} | England {sf2['win2_90']:.1%}")
print(f"  After ET+Penalties: Argentina {sf2['win1_final']:.1%} | England {sf2['win2_final']:.1%}")
print(f"  👉 Winner: {sf2['winner']}")

# Determine finalists and losers
finalist1 = sf1['winner']
finalist2 = sf2['winner']
loser1 = 'France' if finalist1 == 'Spain' else 'Spain'
loser2 = 'England' if finalist2 == 'Argentina' else 'Argentina'

# Final
print("\n" + "-"*60)
print(f"🏆 FINAL: {finalist1} vs {finalist2}")
print("-"*60)
final_match = model.predict(finalist1, finalist2, is_knockout=True)
print(f"  xG: {finalist1} {final_match['xG'][0]:.2f} - {final_match['xG'][1]:.2f} {finalist2}")
print(f"  Most Likely (90 mins): {final_match['most_likely']}")
print(f"  After 90 mins: {finalist1} {final_match['win1_90']:.1%} | Draw {final_match['draw_90']:.1%} | {finalist2} {final_match['win2_90']:.1%}")
print(f"  After ET+Penalties: {finalist1} {final_match['win1_final']:.1%} | {finalist2} {final_match['win2_final']:.1%}")
print(f"  👑 WORLD CUP WINNER: {final_match['winner']}")

# 3rd Place
print("\n" + "-"*60)
print(f"🥉 3RD PLACE: {loser1} vs {loser2}")
print("-"*60)
third = model.predict(loser1, loser2, is_knockout=False)
print(f"  xG: {loser1} {third['xG'][0]:.2f} - {third['xG'][1]:.2f} {loser2}")
print(f"  Most Likely: {third['most_likely']}")
print(f"  {loser1}: {third['win1']:.1%} | Draw: {third['draw']:.1%} | {loser2}: {third['win2']:.1%}")
print(f"  🥉 Winner: {third['winner']}")

# Final Ranking
fourth = loser2 if third['winner'] == loser1 else loser1
print("\n" + "-"*60)
print("🏆 FINAL WORLD CUP RANKING")
print("-"*60)
print(f"  🥇 1st: {final_match['winner']}")
print(f"  🥈 2nd: {finalist2 if final_match['winner'] == finalist1 else finalist1}")
print(f"  🥉 3rd: {third['winner']}")
print(f"  4️⃣ 4th: {fourth}")

print("\n" + "-"*60)

2026 WORLD CUP PREDICTIONS

🏆 SEMI-FINAL 1: Spain vs France
  xG: Spain 2.48 - 2.48 France
  Most Likely (90 mins): 2-2
  After 90 mins: Spain 40.8% | Draw 18.4% | France 40.6%
  After ET+Penalties: Spain 50.0% | France 49.8%
  👉 Winner: Spain
------------------------------------------------------------

🏆 SEMI-FINAL 2: Argentina vs England
  xG: Argentina 2.51 - 2.45 England
  Most Likely (90 mins): 2-2
  After 90 mins: Argentina 41.8% | Draw 18.4% | England 39.5%
  After ET+Penalties: Argentina 51.0% | England 48.8%
  👉 Winner: Argentina

------------------------------------------------------------
🏆 FINAL: Spain vs Argentina
------------------------------------------------------------
  xG: Spain 2.48 - 2.48 Argentina
  Most Likely (90 mins): 2-2
  After 90 mins: Spain 40.6% | Draw 18.4% | Argentina 40.7%
  After ET+Penalties: Spain 49.8% | Argentina 50.0%
  👑 WORLD CUP WINNER: Argentina

------------------------------------------------------------
🥉 3RD PLACE: France vs England
---